# 10 — Prepare UKPN Load Profiles

## Research purpose

Prepare normalised hourly load profiles from the supplied UKPN data to represent daily and seasonal demand patterns. Use these profiles for their temporal shape without transferring the original sites' scale or utilisation to hyperscale campuses.


## Imports

Load the libraries used to clean and summarise the UKPN load profiles.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

## Project location and paths

Locate the project folder and define the paths for the supplied UKPN data, cleaned hourly readings, generic profiles and review table. Check that the required source file is available.


In [ ]:
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "notebooks").is_dir() or not (PROJECT_ROOT / "data").is_dir():
    raise RuntimeError("Launch Jupyter from the folder containing data/ and notebooks/, or from its notebooks/ directory.")

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
INPUT_PATH = DATA_RAW / "demand" / "ukpn-data-centre-demand-profiles.parquet"
CLEANED_HOURLY_OUTPUT_PATH = DATA_INTERIM / "ukpn_datacenter_load_profile_cleaned_hourly.parquet"
GENERIC_PROFILE_OUTPUT_PATH = DATA_INTERIM / "datacenter_generic_load_shape_profiles.csv"
REVIEW_OUTPUT_PATH = DATA_INTERIM / "ukpn_datacenter_load_profile_review.csv"

if not INPUT_PATH.exists():
    raise FileNotFoundError(f"Required supplied UKPN input is missing: {INPUT_PATH}")

## Settings and assumptions

Use the 2023–2025 observations and the Europe/London local clock to build the profiles. Exclude centres with no positive readings or with utilisation above one in more than 5% of their observations.


In [3]:
ACTIVE_WEATHER_YEARS = [2023, 2024, 2025]
PROFILE_TIMEZONE = "Europe/London"
SEVERE_GT1_SHARE_THRESHOLD = 0.05
ZERO_ONLY_MAX_THRESHOLD = 0.0

## Load and clean half-hourly observations

Read the UKPN data using its UTC timestamps and numeric utilisation ratios. Remove invalid records and exact duplicates, retain the selected years and check the remaining centre and year coverage.


In [4]:
required_columns = {
    "cleansed_voltage_level",
    "anonymised_data_centre_name",
    "dc_type",
    "utc_timestamp",
    "hh_utilisation_ratio",
}

ukpn = pd.read_parquet(INPUT_PATH)
missing = sorted(required_columns - set(ukpn.columns))
if missing:
    raise ValueError(f"Missing required UKPN columns: {missing}")

ignored_local_timestamp_columns = sorted(
    column for column in ukpn.columns
    if "local" in column.lower() and "timestamp" in column.lower()
)
ukpn = ukpn[["cleansed_voltage_level", "anonymised_data_centre_name", "dc_type", "utc_timestamp", "hh_utilisation_ratio"]].copy()
ukpn["time_utc"] = pd.to_datetime(ukpn["utc_timestamp"], utc=True, errors="coerce")
ukpn["hh_utilisation_ratio_raw"] = pd.to_numeric(ukpn["hh_utilisation_ratio"], errors="coerce")
ukpn = ukpn.drop(columns=["utc_timestamp", "hh_utilisation_ratio"])

invalid = ukpn["time_utc"].isna() | ukpn["hh_utilisation_ratio_raw"].isna()
if invalid.any():
    print(f"Dropping {invalid.sum():,} rows with invalid timestamp or utilisation ratio")
    ukpn = ukpn[~invalid].copy()

ukpn = ukpn[ukpn["time_utc"].dt.year.isin(ACTIVE_WEATHER_YEARS)].copy()
ukpn = ukpn.drop_duplicates().reset_index(drop=True)
ukpn["weather_year"] = ukpn["time_utc"].dt.year.astype(int)
ukpn["utc_month"] = ukpn["time_utc"].dt.month.astype(int)
ukpn["utc_date"] = ukpn["time_utc"].dt.date.astype(str)
ukpn["utc_hour"] = ukpn["time_utc"].dt.hour.astype(int)
ukpn["day_of_week_utc"] = ukpn["time_utc"].dt.dayofweek.astype(int)
ukpn["utc_day_type"] = np.where(ukpn["day_of_week_utc"].isin([5, 6]), "weekend", "weekday")
ukpn["time_utc_hour"] = ukpn["time_utc"].dt.floor("h")

print(f"Canonical timestamp: utc_timestamp; ignored local timestamp columns: {ignored_local_timestamp_columns or 'none present'}")
print(f"Rows after year filter: {len(ukpn):,}")
print(f"Centres: {ukpn['anonymised_data_centre_name'].nunique():,}")
print("Years:")
print(ukpn["weather_year"].value_counts().sort_index().to_string())
ukpn.head()

Canonical timestamp: utc_timestamp; ignored local timestamp columns: ['local_timestamp']
Rows after year filter: 4,900,274
Centres: 96
Years:
weather_year
2023    1592018
2024    1643620
2025    1664636


,cleansed_voltage_level,anonymised_data_centre_name,dc_type,time_utc,hh_utilisation_ratio_raw,weather_year,utc_month,utc_date,utc_hour,day_of_week_utc,utc_day_type,time_utc_hour
0,Low Voltage Import,Data Centre #74,Co-located,2023-04-29 17:30:00+00:00,0.384134,2023,4,2023-04-29,17,5,weekend,2023-04-29 17:00:00+00:00
1,High Voltage Import,Data Centre #76,Co-located,2023-04-29 17:30:00+00:00,0.424258,2023,4,2023-04-29,17,5,weekend,2023-04-29 17:00:00+00:00
2,Low Voltage Import,Data Centre #77,Co-located,2023-04-29 17:30:00+00:00,0.382033,2023,4,2023-04-29,17,5,weekend,2023-04-29 17:00:00+00:00
3,Low Voltage Import,Data Centre #79,Co-located,2023-04-29 17:30:00+00:00,0.035322,2023,4,2023-04-29,17,5,weekend,2023-04-29 17:00:00+00:00
4,Low Voltage Import,Data Centre #81,Co-located,2023-04-29 17:30:00+00:00,0.408466,2023,4,2023-04-29,17,5,weekend,2023-04-29 17:00:00+00:00


## Review centres and cap utilisation

Summarise each centre's readings and flag centres that fail the inclusion thresholds. Retain eligible centres and cap their half-hourly utilisation ratios between zero and one, keeping the review flags for reference.


In [5]:
centre_summary = (
    ukpn.groupby(["anonymised_data_centre_name", "dc_type", "cleansed_voltage_level"], dropna=False)
    .agg(
        rows=("hh_utilisation_ratio_raw", "size"),
        mean_utilisation_raw=("hh_utilisation_ratio_raw", "mean"),
        max_utilisation_raw=("hh_utilisation_ratio_raw", "max"),
        zero_share=("hh_utilisation_ratio_raw", lambda s: float((s == 0).mean())),
        gt1_share=("hh_utilisation_ratio_raw", lambda s: float((s > 1).mean())),
        gt1_rows=("hh_utilisation_ratio_raw", lambda s: int((s > 1).sum())),
        min_time_utc=("time_utc", "min"),
        max_time_utc=("time_utc", "max"),
    )
    .reset_index()
)
centre_summary["exclude_all_zero"] = centre_summary["max_utilisation_raw"].le(ZERO_ONLY_MAX_THRESHOLD)
centre_summary["exclude_severe_gt1_anomaly"] = centre_summary["gt1_share"].gt(SEVERE_GT1_SHARE_THRESHOLD)
centre_summary["profile_eligible"] = ~(centre_summary["exclude_all_zero"] | centre_summary["exclude_severe_gt1_anomaly"])
centre_summary["review_notes"] = np.select(
    [centre_summary["exclude_all_zero"], centre_summary["exclude_severe_gt1_anomaly"]],
    ["excluded_all_zero_profile", "excluded_gt1_share_above_threshold"],
    default="included_for_profile_construction",
)
eligible_centres = set(centre_summary.loc[centre_summary["profile_eligible"], "anonymised_data_centre_name"])
clean = ukpn[ukpn["anonymised_data_centre_name"].isin(eligible_centres)].copy()
clean["hh_utilisation_ratio_capped"] = clean["hh_utilisation_ratio_raw"].clip(lower=0, upper=1)
clean["ratio_was_capped"] = clean["hh_utilisation_ratio_raw"].gt(1)

print("Centre review:")
print(centre_summary["review_notes"].value_counts().to_string())
print(f"Eligible centres: {len(eligible_centres):,} / {len(centre_summary):,}")
centre_summary.head()

Centre review:
review_notes
included_for_profile_construction     91
excluded_all_zero_profile              4
excluded_gt1_share_above_threshold     1
Eligible centres: 91 / 96


,anonymised_data_centre_name,dc_type,cleansed_voltage_level,rows,mean_utilisation_raw,max_utilisation_raw,zero_share,gt1_share,gt1_rows,min_time_utc,max_time_utc,exclude_all_zero,exclude_severe_gt1_anomaly,profile_eligible,review_notes
0,Data Centre #1,Enterprise,Extra-High Voltage Import,52272,0.028243,0.215803,0.000000,0.000000,0,2023-01-01 00:00:00+00:00,2025-12-31 23:30:00+00:00,False,False,True,included_for_profile_construction
1,Data Centre #10,Co-located,High Voltage Import,52224,0.372789,0.427573,0.000000,0.000000,0,2023-01-01 00:00:00+00:00,2025-12-31 23:30:00+00:00,False,False,True,included_for_profile_construction
2,Data Centre #11,Enterprise,High Voltage Import,52272,0.378055,0.654561,0.000000,0.000000,0,2023-01-01 00:00:00+00:00,2025-12-31 23:30:00+00:00,False,False,True,included_for_profile_construction
3,Data Centre #12,Co-located,High Voltage Import,52272,0.357196,0.544506,0.001454,0.000000,0,2023-01-01 00:00:00+00:00,2025-12-31 23:30:00+00:00,False,False,True,included_for_profile_construction
4,Data Centre #13,Co-located,High Voltage Import,52272,0.342852,1.819675,0.000000,0.003214,168,2023-01-01 00:00:00+00:00,2025-12-31 23:30:00+00:00,False,False,True,included_for_profile_construction


## Aggregate observations to UTC hours

Average the cleaned half-hourly readings for each centre and UTC hour. Retain the number of readings and capped observations in each hourly result.


In [6]:
hourly = (
    clean.groupby(
        [
            "anonymised_data_centre_name",
            "dc_type",
            "cleansed_voltage_level",
            "time_utc_hour",
            "weather_year",
            "utc_month",
            "utc_date",
            "utc_hour",
            "day_of_week_utc",
            "utc_day_type",
        ],
        dropna=False,
    )
    .agg(
        hourly_utilisation_ratio_raw=("hh_utilisation_ratio_raw", "mean"),
        hourly_utilisation_ratio_clean=("hh_utilisation_ratio_capped", "mean"),
        half_hour_records=("hh_utilisation_ratio_capped", "size"),
        capped_half_hour_records=("ratio_was_capped", "sum"),
    )
    .reset_index()
    .rename(columns={"time_utc_hour": "time_utc"})
)

hourly.head()

,anonymised_data_centre_name,dc_type,cleansed_voltage_level,time_utc,weather_year,utc_month,utc_date,utc_hour,day_of_week_utc,utc_day_type,hourly_utilisation_ratio_raw,hourly_utilisation_ratio_clean,half_hour_records,capped_half_hour_records
0,Data Centre #1,Enterprise,Extra-High Voltage Import,2023-01-01 00:00:00+00:00,2023,1,2023-01-01,0,6,weekend,0.009905,0.009905,2,0
1,Data Centre #1,Enterprise,Extra-High Voltage Import,2023-01-01 01:00:00+00:00,2023,1,2023-01-01,1,6,weekend,0.009874,0.009874,2,0
2,Data Centre #1,Enterprise,Extra-High Voltage Import,2023-01-01 02:00:00+00:00,2023,1,2023-01-01,2,6,weekend,0.009911,0.009911,2,0
3,Data Centre #1,Enterprise,Extra-High Voltage Import,2023-01-01 03:00:00+00:00,2023,1,2023-01-01,3,6,weekend,0.009915,0.009915,2,0
4,Data Centre #1,Enterprise,Extra-High Voltage Import,2023-01-01 04:00:00+00:00,2023,1,2023-01-01,4,6,weekend,0.009944,0.009944,2,0


## Add the London local clock

Convert each UTC timestamp to Europe/London and add the local date, month, hour, day type and UTC offset. This conversion accounts for daylight-saving changes.


In [7]:
def add_london_profile_clock(hourly_frame: pd.DataFrame) -> pd.DataFrame:
    """Derive UK civil-clock fields from the canonical UTC timestamp."""
    result = hourly_frame.copy()
    if result["time_utc"].dt.tz is None:
        raise ValueError("time_utc must be timezone-aware UTC before local conversion")
    local = result["time_utc"].dt.tz_convert(PROFILE_TIMEZONE)
    result["time_local_london"] = local
    result["local_date"] = local.dt.date.astype(str)
    result["local_month"] = local.dt.month.astype("int8")
    result["local_hour"] = local.dt.hour.astype("int8")
    result["local_minute"] = local.dt.minute.astype("int8")
    result["local_day_of_week"] = local.dt.dayofweek.astype("int8")
    result["local_day_type"] = np.where(result["local_day_of_week"].isin([5, 6]), "weekend", "weekday")
    local_naive = local.dt.tz_localize(None)
    utc_naive = result["time_utc"].dt.tz_localize(None)
    result["utc_offset_minutes"] = ((local_naive - utc_naive).dt.total_seconds() / 60).astype("int16")
    result["profile_timezone"] = PROFILE_TIMEZONE
    return result

hourly = add_london_profile_clock(hourly)

print(f"Cleaned hourly rows: {len(hourly):,}")
hourly.head()

Cleaned hourly rows: 2,323,226


,anonymised_data_centre_name,dc_type,cleansed_voltage_level,time_utc,weather_year,utc_month,utc_date,utc_hour,day_of_week_utc,utc_day_type,...,capped_half_hour_records,time_local_london,local_date,local_month,local_hour,local_minute,local_day_of_week,local_day_type,utc_offset_minutes,profile_timezone
0,Data Centre #1,Enterprise,Extra-High Voltage Import,2023-01-01 00:00:00+00:00,2023,1,2023-01-01,0,6,weekend,...,0,2023-01-01 00:00:00+00:00,2023-01-01,1,0,0,6,weekend,0,Europe/London
1,Data Centre #1,Enterprise,Extra-High Voltage Import,2023-01-01 01:00:00+00:00,2023,1,2023-01-01,1,6,weekend,...,0,2023-01-01 01:00:00+00:00,2023-01-01,1,1,0,6,weekend,0,Europe/London
2,Data Centre #1,Enterprise,Extra-High Voltage Import,2023-01-01 02:00:00+00:00,2023,1,2023-01-01,2,6,weekend,...,0,2023-01-01 02:00:00+00:00,2023-01-01,1,2,0,6,weekend,0,Europe/London
3,Data Centre #1,Enterprise,Extra-High Voltage Import,2023-01-01 03:00:00+00:00,2023,1,2023-01-01,3,6,weekend,...,0,2023-01-01 03:00:00+00:00,2023-01-01,1,3,0,6,weekend,0,Europe/London
4,Data Centre #1,Enterprise,Extra-High Voltage Import,2023-01-01 04:00:00+00:00,2023,1,2023-01-01,4,6,weekend,...,0,2023-01-01 04:00:00+00:00,2023-01-01,1,4,0,6,weekend,0,Europe/London


## Construct normalised profile variants

Group the hourly readings by London-local month, weekday or weekend, and hour for all eligible sites and for each data-centre type. Divide each bin's mean utilisation by the variant's mean across the profile grid, giving a load shape with mean one. Retain medians, percentiles and observation counts.


In [8]:
def build_normalised_profile_variants(hourly_frame: pd.DataFrame) -> pd.DataFrame:
    """Summarise London-local clock bins and normalise each variant by its grid mean."""
    frames = []
    variants = [("all_clean_sites", hourly_frame)]
    variants.extend(
        (str(dc_type).lower().replace("-", "_").replace(" ", "_"), group)
        for dc_type, group in hourly_frame.groupby("dc_type", dropna=False)
    )
    for variant_name, subset in variants:
        grouped = (
            subset.groupby(["local_month", "local_day_type", "local_hour"], dropna=False)
            .agg(
                mean_utilisation_ratio=("hourly_utilisation_ratio_clean", "mean"),
                median_utilisation_ratio=("hourly_utilisation_ratio_clean", "median"),
                p10_utilisation_ratio=("hourly_utilisation_ratio_clean", lambda s: s.quantile(0.10)),
                p90_utilisation_ratio=("hourly_utilisation_ratio_clean", lambda s: s.quantile(0.90)),
                site_hour_count=("hourly_utilisation_ratio_clean", "size"),
                centre_count=("anonymised_data_centre_name", "nunique"),
            )
            .reset_index()
            .rename(columns={"local_month": "profile_month", "local_day_type": "profile_day_type", "local_hour": "profile_hour"})
        )
        grouped["profile_variant"] = variant_name
        grouped["profile_timezone"] = PROFILE_TIMEZONE
        grouped["variant_mean_utilisation_ratio"] = subset["hourly_utilisation_ratio_clean"].mean()
        frames.append(grouped)

    result = pd.concat(frames, ignore_index=True)
    grid_mean = result.groupby("profile_variant")["mean_utilisation_ratio"].transform("mean")
    result["profile_grid_mean_utilisation_ratio"] = grid_mean
    result["normalized_load_shape"] = result["mean_utilisation_ratio"] / grid_mean.where(grid_mean.ne(0), np.nan)
    ordered = [
        "profile_variant", "profile_timezone", "profile_month", "profile_day_type", "profile_hour",
        "normalized_load_shape", "mean_utilisation_ratio", "median_utilisation_ratio",
        "p10_utilisation_ratio", "p90_utilisation_ratio", "variant_mean_utilisation_ratio",
        "profile_grid_mean_utilisation_ratio", "site_hour_count", "centre_count",
    ]
    return result[ordered].sort_values(["profile_variant", "profile_month", "profile_day_type", "profile_hour"]).reset_index(drop=True)

profiles = build_normalised_profile_variants(hourly)
print(f"Profile variants: {sorted(profiles['profile_variant'].unique().tolist())}")
print(profiles.head(8).to_string(index=False))

Profile variants: ['all_clean_sites', 'co_located', 'enterprise']
profile_variant profile_timezone  profile_month profile_day_type  profile_hour  normalized_load_shape  mean_utilisation_ratio  median_utilisation_ratio  p10_utilisation_ratio  p90_utilisation_ratio  variant_mean_utilisation_ratio  profile_grid_mean_utilisation_ratio  site_hour_count  centre_count
all_clean_sites    Europe/London              1          weekday             0               0.999271                0.209409                  0.190501               0.000797               0.415378                        0.210903                             0.209562             6005            91
all_clean_sites    Europe/London              1          weekday             1               0.998457                0.209238                  0.189796               0.000627               0.415140                        0.210903                             0.209562             6005            91
all_clean_sites    Europe/London        

## Check the London clock conversion

Check the local dates, hours, offsets and day types for selected UTC timestamps. These examples cover winter, summer and both daylight-saving transitions.


In [9]:
def validate_london_clock_conversion() -> None:
    """Check standard time, DST skip/repeat, date, leap-day, and weekend semantics."""
    tests = pd.DataFrame(
        {
            "time_utc": pd.to_datetime([
                "2024-01-15 12:00:00Z",
                "2024-07-15 12:00:00Z",
                "2024-03-31 00:00:00Z",
                "2024-03-31 01:00:00Z",
                "2024-10-27 00:00:00Z",
                "2024-10-27 01:00:00Z",
                "2024-03-01 00:00:00Z",
            ], utc=True),
        }
    )
    actual = add_london_profile_clock(tests)
    expected = [
        ("2024-01-15", 12, 0, 0, "weekday"),
        ("2024-07-15", 13, 0, 60, "weekday"),
        ("2024-03-31", 0, 0, 0, "weekend"),
        ("2024-03-31", 2, 0, 60, "weekend"),
        ("2024-10-27", 1, 0, 60, "weekend"),
        ("2024-10-27", 1, 0, 0, "weekend"),
        ("2024-03-01", 0, 0, 0, "weekday"),
    ]
    observed = list(actual[["local_date", "local_hour", "local_minute", "utc_offset_minutes", "local_day_type"]].itertuples(index=False, name=None))
    if observed != expected:
        raise ValueError(f"Europe/London clock validation failed: {observed}")

validate_london_clock_conversion()

## Validate and export the profiles

Check utilisation bounds, year coverage, unique centre-hour records and the complete local-clock grid for each profile variant. Confirm that every normalised profile has mean one and inspect the utilisation distributions. Save the centre review, cleaned hourly readings and generic profiles.


In [10]:
if hourly["hourly_utilisation_ratio_clean"].lt(0).any() or hourly["hourly_utilisation_ratio_clean"].gt(1).any():
    raise ValueError("Cleaned hourly utilisation ratios must be within [0, 1]")

required_profile_variants = {"all_clean_sites"} | {
    str(dc_type).lower().replace("-", "_").replace(" ", "_")
    for dc_type in hourly["dc_type"].unique()
}
missing_variants = sorted(required_profile_variants - set(profiles["profile_variant"]))
if missing_variants:
    raise ValueError(f"Missing expected profile variants: {missing_variants}")

if sorted(ukpn["weather_year"].unique().tolist()) != ACTIVE_WEATHER_YEARS:
    raise ValueError("UKPN input must cover all configured weather years")
if hourly.duplicated(["anonymised_data_centre_name", "time_utc"]).any():
    raise ValueError("Cleaned hourly output contains duplicate centre/UTC-hour keys")
if not hourly["local_minute"].eq(0).all():
    raise ValueError("Europe/London hourly conversion produced non-zero local minutes")
if set(hourly["utc_offset_minutes"].unique()) != {0, 60}:
    raise ValueError("Unexpected Europe/London UTC offsets")

expected_grid = pd.MultiIndex.from_product(
    [sorted(required_profile_variants), range(1, 13), ["weekday", "weekend"], range(24)],
    names=["profile_variant", "profile_month", "profile_day_type", "profile_hour"],
)
actual_grid = pd.MultiIndex.from_frame(profiles[list(expected_grid.names)])
if actual_grid.has_duplicates or set(actual_grid) != set(expected_grid):
    raise ValueError("Local-clock profile grid is incomplete or duplicated")
grid_means = profiles.groupby("profile_variant")["normalized_load_shape"].mean()
if not np.allclose(grid_means.to_numpy(), 1.0, atol=1e-12, rtol=0):
    raise ValueError(f"Normalised profile grid means must equal 1: {grid_means.to_dict()}")
profile_grid = profiles.groupby("profile_variant").size()

print("Profile rows by variant:")
print(profile_grid.to_string())
print("Cleaned hourly utilisation summary:")
print(hourly["hourly_utilisation_ratio_clean"].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).to_string())
print("Profile normalized shape summary:")
print(profiles.groupby("profile_variant")["normalized_load_shape"].describe().to_string())

centre_summary.to_csv(REVIEW_OUTPUT_PATH, index=False)
hourly.to_parquet(CLEANED_HOURLY_OUTPUT_PATH, index=False, compression="zstd")
profiles.to_csv(GENERIC_PROFILE_OUTPUT_PATH, index=False)

Profile rows by variant:
profile_variant
all_clean_sites    576
co_located         576
enterprise         576
Cleaned hourly utilisation summary:
count    2.323226e+06
mean     2.109027e-01
std      1.721511e-01
min      0.000000e+00
1%       0.000000e+00
5%       0.000000e+00
50%      1.918851e-01
95%      5.411204e-01
99%      6.785455e-01
max      1.000000e+00
Profile normalized shape summary:
                 count  mean       std       min       25%       50%       75%       max
profile_variant                                                                         
all_clean_sites  576.0   1.0  0.030567  0.944542  0.979200  0.997122  1.016997  1.085267
co_located       576.0   1.0  0.029369  0.950542  0.979552  0.994547  1.018319  1.083029
enterprise       576.0   1.0  0.042864  0.908700  0.969134  1.001019  1.028984  1.102152
